# scRNA-seq 中的可变 polyadenylation 分析

Alternative polyadenylation（APA，可变多聚腺苷酸化）是信使 RNA（mRNA）成熟过程中发生的一类生物学过程：同一个基因内部可以使用不同的 polyadenylation signals。这意味着同一段 DNA 序列可以产生多个 3' 端长度不同的 mRNA 版本。这些变化会影响 RNA 稳定性、细胞内定位，以及 RNA 被翻译成蛋白质的方式。因此，APA 在基因表达调控中具有重要作用，也可能与细胞分化、癌症等疾病过程相关。

[SCAPE-APA](https://github.com/chengl7-lab/scape)（Single Cell Analysis of Polyadenylation Events）是一个用于在 scRNA-seq 数据中识别和量化 APA 事件的计算工具包。它可以帮助研究者分析不同细胞如何在基因中使用不同的 polyadenylation sites，从而揭示经典基因表达分析难以捕捉的额外调控层次。

在本 notebook 中，我们将在 Google Colab 环境中使用 SCAPE-APA，从 scRNA-seq 数据估计 APA 事件。目标是提供一个实用、直观的入门示例，展示 APA 如何在单个细胞层面贡献调控多样性。


### Conda

Conda 是一个常用于数据科学和生物信息学的环境与包管理工具。它可以方便地安装软件和库，即使这些软件具有复杂依赖关系。

Conda 虚拟环境是一个隔离空间，你可以在其中安装特定版本的软件包，而不会干扰系统中的其他安装。这可以避免依赖冲突，并提高分析的可重复性。


In [ ]:
# Installing the condalab package, allowing you to use Conda in Google Colab, along with its packages and features.
!pip install -q condacolab
import condacolab
condacolab.install()

等待该步骤运行完成，并在会话重启后再继续。


In [ ]:
%%capture
#Installation of numerical packages and tools for data visualization.
!conda install -y -c conda-forge -c bioconda -c defaults \
    numpy scipy pandas matplotlib \
    click tomli-w requests psutil \
    bedtools pybedtools pysam gffutils

#Install scape
!pip install taichi scape-apa
!apt-get install -y samtools
%env MPLBACKEND=Agg

In [ ]:
# Verifying SCAPE-APA installation.
!scape --help

### GitHub

GitHub 是一个在线平台，开发者可以在上面使用 Git 版本控制系统存储、共享和协作开发代码项目。

`git clone` 命令用于把 GitHub 或其他 Git 服务器上的完整代码仓库复制到本地。它会下载所有文件、版本历史和项目结构，使你能够在本地继续操作。下面以 SCAPE-APA 仓库为例执行这一过程。


In [ ]:
# Getting the files from the GitHub repository.
!git clone https://github.com/chengl7-lab/scape.git

In [ ]:
#List of files inside the "toy example" folder.
!ls -lh scape/examples/toy-example/

In [ ]:
# Download annotation file "Mus_musculus.GRCm39.113.chr.gff3"
# wget is for downloading
# -O to define where the file goes and what name
!wget -O scape/examples/toy-example/Mus_musculus.GRCm39.113.chr.gff3.gz https://ftp.ensembl.org/pub/release-113/gff3/mus_musculus/Mus_musculus.GRCm39.113.chr.gff3.gz

In [ ]:
# Uncompress file
!gunzip  scape/examples/toy-example/Mus_musculus.GRCm39.113.chr.gff3.gz

In [ ]:
# Checking internal content.
!head -n 20 scape/examples/toy-example/Mus_musculus.GRCm39.113.chr.gff3

In [ ]:
# check file lines
!wc -l scape/examples/toy-example/Mus_musculus.GRCm39.113.chr.gff3

# **UTR 区域注释**

UTR 是 mRNA 的 untranslated regions（非翻译区），位于基因的 5' 端和 3' 端。APA 通常发生在 3' 端，也就是 3'UTR 内。因此，正确界定这些区域的位置，对于准确检测一个基因中可能发生的不同 APA 事件至关重要。

`gen_utr_annotation` 命令是 SCAPE-APA 分析流程的一部分。它的主要目标是从 `.gff3` 格式文件生成 UTR（untranslated regions）区域注释；`.gff3` 文件中包含某一物种的基因组注释信息。

* 这个命令实际做什么？

1. 读取 `.gff3` 文件，其中包含基因结构信息。

2. 基于这些注释，识别并提取每个 transcript 的 3'UTR 区域。

3. 生成包含这些区域的输出文件。SCAPE-APA 后续会使用该文件检测不同 polyadenylation signals 出现的位置。


#### **注意：** 该步骤在 GPU 上大约需要 1 小时，在 CPU 上大约需要 2 小时。如果使用示例文件，可以跳过这一步。


In [ ]:
# scape gen_utr_annotation = Call SCAPE with the "gen_utr_annotation" command.
# --gff_file = ".gff3" file.
# --output_dir = Output path of the generated file.
# --res_file_name = Name of the generated file, resulting in ".csv" format.

!scape gen_utr_annotation \
--gff_file scape/examples/toy-example/Mus_musculus.GRCm39.113.chr.gff3 \
--output_dir scape/examples/toy-example/ \
--res_file_name example_annotation

运行下面的命令，获取可直接使用的输出文件。


In [ ]:
!wget -O scape/examples/toy-example/example_annotation.csv "https://raw.githubusercontent.com/integrativebioinformatics/scNotebooks/refs/heads/main/scNotebooks-Resources/example_annotation.csv"

# **准备 scRNA-seq 数据**

`prepare_input` 命令是 SCAPE-APA 流程中的关键步骤，用于为检测 alternative polyadenylation（APA）事件准备 scRNA-seq 数据。

该步骤只关注位于 UTR 区域中的 reads，因此可以降低噪声，并把分析聚焦在 APA 事件发生的区域。这会提高检测的准确性和效率，使我们能够更可靠地估计单个细胞中不同 polyadenylation sites 的使用情况。

* 该步骤做什么？

1. 处理 `.bam` 文件；该文件包含单细胞 RNA-seq reads 与基因组的比对结果。

2. 过滤并选择那些比对到 UTR 区域的 reads，主要是前一步 `gen_utr_annotation` 命令注释得到的 3'UTR。

3. 将相关数据整理为适合后续 SCAPE-APA 分析的格式。


#### **注意：** 该步骤在 GPU 上大约需要 7 分钟。如果使用示例文件，可以跳过这一步。


In [ ]:
# scape prepare_input = Command to prepare data from a ".bam" file.
# --utr_file = Path to the UTR annotation file generated above.
# --cb_file = Barcode file to identify cells in scRNA-seq data.
# --bam_file = ".bam" file of aligned reads.

!scape prepare_input --utr_file scape/examples/toy-example/example_annotation.csv \
--cb_file scape/examples/toy-example/barcodes.tsv.gz \
--bam_file scape/examples/toy-example/example.bam \
--output_dir scape/examples/toy-example/

In [ ]:
# check directory
!ls -lh scape/examples/toy-example/

### Samtools

*Samtools* 是一组命令行工具，用于处理序列比对文件，尤其是 SAM（Sequence Alignment/Map）和 BAM（SAM 的压缩版本）格式。

下面安装 Samtools：


In [ ]:
# Viewing ".bam" file with samtools.

!samtools view scape/examples/toy-example/example.bam | head -n 10

Samtools 可以为 BAM 文件创建索引，使生物信息学工具能够快速访问该文件中特定的基因组区域。例如，在基因组浏览器中查看比对结果，或在分析中提取特定区域时，BAM 索引都是必要的。


In [ ]:
# Make index
!samtools index scape/examples/toy-example/example.bam

In [ ]:
#Lists all files in the specified directory
# The pipe (|) is an operator used to chain commands,
# allowing the output of one command to be used directly as input for the next
# | grep ".pkl" = filters the output of the previous command and shows only the lines that contain files with the .pkl extension

!ls -lh scape/examples/toy-example/ | grep ".pkl"

# **推断 APA 事件**

使用 SCAPE-APA 可以推断细胞如何使用不同的 RNA 终止信号；这是理解单细胞层面基因调控的重要过程。检测这些事件可以帮助我们研究更精细的表达模式，以及它们对生物过程或疾病可能产生的影响。

这里会在前面注释好的 3'UTR 区域上使用 `infer_pa`。

* 该步骤做什么？

1. 分析输入数据，也就是在 `prepare_input` 步骤中过滤得到的数据，以检测 UTR 区域内的 alternative polyadenylation sites（APA sites）。

2. 识别同一基因的不同 transcripts 在哪些位置发生 polyadenylation，从而推断转录后调控 isoforms 的多样性。


In [ ]:
# Command to run the inference model and detect polyadenylation sites.
!scape infer_pa \
--pkl_input_file scape/examples/toy-example/pkl_input/example.100.1.1.input.pkl \
--output_dir scape/examples/toy-example/ \
--toml_para_file scape/tutorial/default_config.toml

执行 `infer_pa` 命令。

该命令负责基于 scRNA-seq 比对结果和前面注释好的 UTR 区域，推断 alternative polyadenylation（APA）事件。

主要命令参数：

* `utr_file`：包含 UTR 区域注释的 `.csv` 文件，由 `gen_utr_annotation` 步骤生成。

* `cb_file`：包含 cell barcodes 的文件，用于逐个识别细胞。

* `bam_file`：包含 RNA reads 基因组比对结果的 `.bam` 文件。

* `output_dir`：保存输出文件的目录。

主要推断参数：

* `chunksize`：每次分析的 UTR 数量，有助于在大数据集中管理内存。

* `n_max_apa` / `n_min_apa`：算法在每个 UTR 中检测的 APA sites 最大和最小数量。

* `min_LA` / `max_LA`：alignment regions 允许的最小和最大大小，用于调节噪声与精度。

* `mu_f` / `sigma_f`：RNA fragment length 的均值和标准差，是数据建模的重要参数。

* `min_pa_gap`：两个不同 APA sites 之间的最小距离，用于避免检测到距离过近的假阳性位点。

* `max_beta`、`beta_step`：控制 beta distribution，该分布用于 APA sites 位置的统计建模。

* `theta_step`：theta 值范围，是混合分布中的另一个参数。

* `min_ws` / `max_unif_ws`：控制建模中各分布的权重，用于区分真实事件和噪声。

* `re_run_mode`：如果启用（TRUE），允许覆盖先前结果并重新运行分析。

* `fixed_run_mode`：如果禁用（FALSE），允许算法自动调整参数，以获得更好的推断结果。


# **合并事件**

`merge_pa` 命令用于合并 SCAPE-APA 前面步骤中检测到的 APA 事件，把属于同一基因或同一 UTR 区域的结果整合起来。

* 实际做什么？

1. 接收按细胞或按 UTR 区域推断得到的 APA 事件结果作为输入，这些结果来自 `infer_pa`。

2. 将发生在同一基因组区域的 APA sites 进行分组，例如同一个 3'UTR 或同一个基因内部的位点。

3. 去除冗余，并为每个基因生成最具代表性的切割位点汇总。

在推断过程中，不同细胞或重复样本中可能检测到多个 APA sites。`merge_pa` 函数可以把相似事件统一起来，便于解释。此外，它还能降低统计噪声，并帮助整体可视化每个基因的 APA 模式；这些结果可以继续用于聚类、可视化，或与细胞 metadata 关联等后续分析。


In [ ]:
# With "merge_pa", the polyadenylation sites inferred in the previous step are collected,
# #merging the sites within the same gene or UTR.

# Generates two "res.TYPE.pk1" files, where "TYPE" can be the gene sites
# or merged UTRs (gene - UTR).

!scape merge_pa --output_dir scape/examples/toy-example/

In [ ]:
# List files in toy-example
!ls -lh scape/examples/toy-example/

In [ ]:
# SCAPE-APA analysis to calculate the effective length of detected APA events

!scape cal_exp_pa_len \
--output_dir scape/examples/toy-example \
--res_pkl_file res.gene.pkl

In [ ]:
# List files in toy-example
!ls -lh scape/examples/toy-example/

# **查看结果**


### `gen_utr_annotation`


Pandas 是 Python 中功能强大的数据处理与分析库。借助 Pandas，可以像操作 Excel 表格或 SQL 表一样处理表格数据（DataFrames）。下面用它来检查数据。


In [ ]:
import pandas as pd

# Load the UTR annotation
df_utr = pd.read_csv("scape/examples/toy-example/example_annotation.csv")

# Display the first few rows of the table
df_utr.head()

In [ ]:
import matplotlib.pyplot as plt

# Calculate the length of each UTR
df_utr["length"] = df_utr["end"] - df_utr["start"]

# Plot a length histogram
plt.figure(figsize=(8, 5))
plt.hist(df_utr["length"], bins=30, edgecolor='black')
plt.xlabel("UTR length (bp)")
plt.ylabel("Frequency")
plt.title("UTR Length distribution")
plt.show()

In [ ]:
# Run merge_pa
!scape merge_pa --output_dir scape/examples/toy-example \
--utr_merge True

In [ ]:
# Check if "prepare_input" worked
# Expect ".pk1" files

import os

# List files
os.listdir("scape/examples/toy-example/pkl_input/")

In [ ]:
# Displaying parameters by "infer_pa" (res.utr.pkl)
import pickle
f=open("scape/examples/toy-example/res.utr.pkl", "rb")
print(pickle.load(f))

In [ ]:
import matplotlib.pyplot as plt

# PA Site Data
pa_sites = ["PA1 (2965)", "PA2 (4171)"]
usage_probs = [0.16, 0.84]

# Create Bar Chart
plt.figure(figsize=(6, 4))
plt.bar(pa_sites, usage_probs, color=['blue', 'red'])
plt.xlabel("Polyadenylation Sites")
plt.ylabel("Usage Probability")
plt.title("Polyadenylation Site Preference in the UTR")
plt.ylim(0, 1)

# Show plot
plt.show()

In [ ]:
# merge_pa
# review data
with open("scape/examples/toy-example/res.gene.pkl", "rb") as f:
    try:
        merged_pa = pickle.load(f)
        print(merged_pa)
    except EOFError:
        print("The file res.gene.pkl is empty.")

In [ ]:
# cal_exp_pa_len
df_exp_len = pd.read_csv("scape/examples/toy-example/cluster_wrt_CB.gene.pa.len.csv")

# Show first lines
df_exp_len.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_exp_len.iloc[:, 1], bins=30, edgecolor='black')
plt.xlabel("Expected Polyadenylation Length (bp)")
plt.ylabel("Freq")
plt.title("Distribution of Expected Polyadenylation Length")
plt.show()

In [ ]:
#ex_pa_cnt_mat
df_counts = pd.read_csv("scape/examples/toy-example/res.utr.cnt.tsv.gz", sep="\t")

# Show fisrt lines
df_counts.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_counts.iloc[:, 1], bins=30, edgecolor='black')
plt.xlabel("Number of Polyadenylation Events")
plt.ylabel("Frequency")
plt.title("Distribution of Polyadenylation Events")
plt.show()